# Steerling-8B: Chunk-to-Concept Attribution

For each predicted token, Steerling's concept heads produce per-concept contributions to the
output logit. To summarize an entire chunk, we **sum absolute contributions** across all tokens
in the chunk and report each concept's fraction of total logit mass:

$$\text{chunk}_\text{score}(i) = \sum_{t \in \text{chunk}} |w_i \cdot (e_i \cdot W_{y_t})|$$

$$\text{fraction}(i) = \frac{\text{chunk}_\text{score}(i)}{\sum_{\text{all concepts}} \text{chunk}_\text{score} + \sum_t |\varepsilon_t|}$$

Epsilon is included in the denominator (shown as a percentage in the title) but not plotted.

**Requirements:** Interpretable Steerling model, GPU with >= 18 GB VRAM

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from pathlib import Path
from transformers import AutoModel, AutoTokenizer
from steerling import SteerlingGenerator, GenerationConfig

PURPLE = "#675BF2"
PURPLE_LIGHT = "#ceb4fe"
GOLD = "#e2bc26"
COLOR_MAP = {"known": PURPLE, "discovered": PURPLE_LIGHT}

## 1. Load Model & Concept Labels

Load the interpretable Steerling model and the CSV mapping of known concept IDs to
human-readable names. Discovered concepts (101,196) have no labels and are shown as numeric IDs.

In [ ]:
model_id = "guidelabs/steerling-8b"

model = AutoModel.from_pretrained(model_id, trust_remote_code=True, dtype=torch.bfloat16)
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
generator = SteerlingGenerator.from_model(model, tokenizer, device="cuda")

backbone = generator.model.model if hasattr(generator.model, "model") else generator.model
device = generator.device

print(generator)
print(f"Interpretable: {generator.is_interpretable}")

In [ ]:
def find_repo_root(marker="pyproject.toml"):
    for p in [Path().resolve(), *Path().resolve().parents]:
        if (p / marker).exists():
            return p
    raise FileNotFoundError(f"Could not find repo root (looking for {marker})")

REPO_ROOT = find_repo_root()
concepts_path = REPO_ROOT / "assets" / "concepts" / "known_concepts.csv"

if concepts_path.exists():
    concepts_df = pd.read_csv(concepts_path)
    print(f"Loaded {len(concepts_df)} concept labels")
else:
    concepts_df = pd.DataFrame(columns=["concept_idx", "concept_name"])
    print("No concept file found, using fallback labels")


def concept_label(concept_id: int, concept_type: str = "known") -> str:
    if concept_type == "discovered":
        return f"Discovered #{concept_id}"
    row = concepts_df[concepts_df["concept_idx"] == concept_id]
    if len(row) == 0:
        return f"Known #{concept_id}"
    return row.iloc[0]["concept_name"]

## 2. Generate & Forward Pass


In [ ]:
prompt = "AI technology will"
config = GenerationConfig(max_new_tokens=128, steps=128, temperature=0.4)

gen_output = generator.generate_full(prompt, config)
output = gen_output.tokens.unsqueeze(0)  # (1, L) for backbone forward pass

# Forward pass — minimal_output=True is sufficient since we only need sparse top-k
with torch.no_grad():
    logits, outputs = backbone(output, minimal_output=True)

print("Generated text:")
print(gen_output.text)

## 3. Compute Chunk Attribution

For each token, we compute how much each active concept pushes toward the predicted token:
`contribution = sigmoid(logit) * (concept_embedding · lm_head_weight)`. We then normalize
per-token and aggregate by concept ID using `scatter_add` (since different tokens may have
different concepts in their top-k slots). Finally, we average over the chunk length T.

In [ ]:
@torch.no_grad()
def compute_concept_attribution(outputs, logits, backbone):
    """
    Per-position concept contributions using sparse top-k indices.

    Note: For 'known' concepts, the display label comes from the CSV file if it exists,
    via the concept_label() function. If the CSV does not contain a name for a given concept_id,
    a fallback label like 'Known #{id}' is used. See concept_label() definition for details.
    """
    pred_ids = logits.argmax(dim=-1)
    pred_logits = torch.gather(logits, -1, pred_ids.unsqueeze(-1)).squeeze(-1)
    W_y = backbone.transformer.lm_head.weight[pred_ids]

    def _head_contrib(head, topk_idx=None, topk_logits=None):
        """Compute contribution from sparse top-k indices."""
        if topk_idx is None:
            return None
        emb = head._get_embedding(topk_idx)
        dots = torch.einsum("btkd,btd->btk", emb, W_y)
        w = torch.sigmoid(topk_logits.float())
        return topk_idx, w * dots

    known = _head_contrib(
        backbone.known_head,
        topk_idx=outputs.known_topk_indices,
        topk_logits=outputs.known_topk_logits,
    )

    disc = None
    if getattr(backbone, "unknown_head", None) is not None:
        disc = _head_contrib(
            backbone.unknown_head,
            topk_idx=outputs.unknown_topk_indices,
            topk_logits=outputs.unknown_topk_logits,
        )

    bsz, tlen = pred_ids.shape
    zero = lambda: (
        torch.zeros(bsz, tlen, 0, dtype=torch.long, device=device),
        torch.zeros(bsz, tlen, 0, device=device),
    )

    k_idx, k_c = known if known else zero()
    d_idx, d_c = disc if disc else zero()
    eps = pred_logits - k_c.sum(-1) - d_c.sum(-1)

    return {
        "known_indices": k_idx, "known_contributions": k_c,
        "disc_indices": d_idx, "disc_contributions": d_c,
        "epsilon": eps,
    }


def find_chunks(token_ids):
    """Split token IDs at <|endofchunk|> boundaries."""
    ids = token_ids.squeeze()
    eoc_id = getattr(tokenizer, "endofchunk_token_id", None)
    if eoc_id is None:
        return [(0, len(ids))]

    positions = (ids == eoc_id).nonzero(as_tuple=True)[0].tolist()
    chunks, prev = [], 0
    for p in positions:
        if p > prev:
            chunks.append((prev, p))
        prev = p + 1
    if prev < len(ids):
        chunks.append((prev, len(ids)))
    return chunks


def chunk_attribution(attr, start, end, batch=0):
    """
    Compute concept attribution over a chunk of tokens.

    For "known" concepts, the label comes from the concept CSV file (if present)
    via the concept_label() function. If the CSV does not contain the id,
    a fallback label is used.

    Per position, each concept's contribution is normalized by the total absolute
    logit mass at that position (known + discovered + epsilon). This removes
    scale differences across positions so every token votes equally to the chunk.
    """

    k_idx = attr["known_indices"][batch, start:end]        # [T, K_known]
    k_c   = attr["known_contributions"][batch, start:end]  # [T, K_known]
    d_idx = attr["disc_indices"][batch, start:end]         # [T, K_disc]
    d_c   = attr["disc_contributions"][batch, start:end]   # [T, K_disc]
    eps   = attr["epsilon"][batch, start:end]              # [T] residual

    pos_total = (
        k_c.abs().sum(-1) + d_c.abs().sum(-1) + eps.abs()
    ).clamp(min=1e-8)                                      # [T]

    k_c_norm = k_c / pos_total.unsqueeze(-1)              # [T, K_known]
    d_c_norm = d_c / pos_total.unsqueeze(-1)              # [T, K_disc]

    def aggregate(idx, contrib_norm):
        """
        Accumulate signed per-position fractions into per-concept attribution.
        """
        flat_idx  = idx.reshape(-1)           # [T*K]
        flat_norm = contrib_norm.reshape(-1)  # [T*K]
        unique_ids, inverse = flat_idx.unique(return_inverse=True)
        mass = torch.zeros(len(unique_ids), device=flat_idx.device)
        mass.scatter_add_(0, inverse, flat_norm)
        return unique_ids, mass               # [num_unique_concepts]

    k_ids, k_mass = aggregate(k_idx, k_c_norm)
    d_ids, d_mass = aggregate(d_idx, d_c_norm)

    # Divide by T so contribution is in normal range
    T = end - start

    entries = []
    for cid, mass in zip(k_ids.tolist(), k_mass.tolist()):
        entries.append({
            "label": concept_label(int(cid), "known"),   # from CSV if available
            "type": "known",
            "contribution": mass / T,
        })
    for cid, mass in zip(d_ids.tolist(), d_mass.tolist()):
        entries.append({
            "label": f"Discovered #{int(cid)}",
            "type": "discovered",
            "contribution": mass / T,
        })

    eps_pct = float((eps / pos_total).mean()) * 100

    return entries, eps_pct


# Compute
attr = compute_concept_attribution(outputs, logits, backbone)
chunks = find_chunks(output)

print(f"Found {len(chunks)} chunks:\n")
for i, (s, e) in enumerate(chunks):
    text = tokenizer.decode(output[0, s:e].tolist())
    print(f"  Chunk {i}: [{s}, {e}) — {e - s} tokens")
    print(f"    {text!r}\n")

## 4. Plot

Top-20 concepts per chunk as horizontal bar charts. Known concepts in purple, discovered in
light purple. Epsilon (residual unexplained by concepts) is shown as a percentage in the title.

In [ ]:
def plot_contributions(entries, title="Concept Contributions", eps_pct=0.0, top_k=20):
    """
    Horizontal bar chart of top concept contributions.

    Note: The labels shown on the plot are taken from entry['label'] for each entry in `entries`.
    The source of these labels depends on how `entries` is constructed:
      - For "known" concepts, the label is provided by the function `concept_label()`, which may
        internally use a CSV or mapping to resolve concept ids to human-friendly names.
      - For "discovered" concepts, the label is generated as a string like "Discovered #<id>".

    If `concept_label()` looks up labels from a CSV, then those will appear here. Check the
    implementation of `concept_label()` for details.
    """
    display = sorted(entries, key=lambda e: e["contribution"], reverse=True)[:top_k]

    labels = [e["label"] for e in display]
    values = [e["contribution"] for e in display]
    colors = [COLOR_MAP[e["type"]] for e in display]

    fig, ax = plt.subplots(figsize=(10, max(3, len(labels) * 0.35)))
    y = np.arange(len(labels))
    ax.barh(y, values, color=colors)
    ax.set_yticks(y)
    ax.set_yticklabels(labels, fontsize=9)
    ax.invert_yaxis()
    ax.set_xlabel("Fraction of total |logit| mass")
    ax.set_title(f"{title}  |  epsilon = {eps_pct:.1f}%", fontsize=11)
    ax.legend(
        handles=[Patch(fc=COLOR_MAP[t], label=t.title()) for t in ["known", "discovered"]],
        loc="lower right", fontsize=8,
    )
    # Remove the top and right axes (spines)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()
    plt.show()

In [ ]:
for i, (s, e) in enumerate(chunks[:2]):
    text_preview = tokenizer.decode(output[0, s:e].tolist())[:100]
    entries, eps_pct = chunk_attribution(attr, s, e)

    print(f"\n=== Chunk {i}: [{s}, {e}) | epsilon = {eps_pct:.1f}% ===")
    print(f"  Text: {text_preview!r}...")
    top = sorted(entries, key=lambda e: e["contribution"], reverse=True)[:10]
    for e in top:
        print(f"  {e['type']:<12s} {e['contribution']:6.2%}  {e['label']}")

    plot_contributions(entries, title=f'Chunk {i}: "{text_preview}..."', eps_pct=eps_pct, top_k=20)